# CODICE PULITO di visualizzazione_dati_test
1. modifiche file singoli file merge (es. conversioni dummy, round age)
2. merge outer
3. creazione colonna globale per variabile comune ai due dataset
4. studio e applicazione strategia dove ci sono le differenze
5. ricordati ritrasformare dummy e cancellare x e y

- appuntino --> age precedenza PTDEMOG
- da fare prima del merge ---> cancellare "AGE_bl" + dedummyzare
- sistemare "DX" PTDEMOG
- considerare se possibile cancellare prima del merge "VISCODE" == "f"


In [2]:
import pandas as pd
import numpy as np

In [3]:
# --- 1. Caricamento dei dataset puliti ---
adnimerge = pd.read_csv("ADNIMERGE_cleaned_02.csv", parse_dates=["EXAMDATE", "update_stamp"])
ptdemog = pd.read_csv("PTDEMOG_cleaned_02.csv", parse_dates=["EXAMDATE", "update_stamp", "PTDOB"])

## Riconversione dummys

In [4]:
cols_x = sorted([c for c in ptdemog.columns if c.endswith('_0')])
coppie = []
for c in cols_x:
    base = c[:-2]
    c_y = base + '_1'
    if c_y in ptdemog.columns:
        coppie.append(base)

print("Coppie trovate:", coppie)

Coppie trovate: ['ETHNICITY', 'GENDER', 'MARRY', 'RACE']


In [5]:
# --- ADNIMERGE ---
"""
for prefix, categories in [
    ("GENDER", [0, 1]),
    ("MARRY", [0, 1, 2, 3]),
    ("ETHNICITY", [0, 1]),
    ("RACE", [0, 1, 2, 3, 4, 5]),
]:
    cols_dummy = [f"{prefix}_{cat}" for cat in categories]
    ricostruita = pd.from_dummies(adnimerge[cols_dummy], sep="_", default_category="missing")
    adnimerge[prefix] = ricostruita[prefix]
    adnimerge = adnimerge.drop(columns=cols_dummy)"""

'\nfor prefix, categories in [\n    ("GENDER", [0, 1]),\n    ("MARRY", [0, 1, 2, 3]),\n    ("ETHNICITY", [0, 1]),\n    ("RACE", [0, 1, 2, 3, 4, 5]),\n]:\n    cols_dummy = [f"{prefix}_{cat}" for cat in categories]\n    ricostruita = pd.from_dummies(adnimerge[cols_dummy], sep="_", default_category="missing")\n    adnimerge[prefix] = ricostruita[prefix]\n    adnimerge = adnimerge.drop(columns=cols_dummy)'

In [6]:
# --- PTDEMOG ---
"""def ricostruisci_categorie(df, lista_categorie):
    cols_dummy = []
    for prefix in lista_categorie:#["GENDER", "MARRY", "ETHNICITY", "RACE"]:
        cols = df.columns[df.columns.str.startswith(prefix)].tolist()
        cols_dummy.extend(cols)
        
        ricostruita = pd.from_dummies(df[cols_dummy], sep="_", default_category="__MISSING__").replace("__MISSING__", np.nan)
        df[prefix] = ricostruita[prefix]
    df = df.drop(columns=cols_dummy)
    return df

adnimerge = ricostruisci_categorie(adnimerge, ['ETHNICITY', 'GENDER', 'MARRY', 'RACE'])
ptdemog = ricostruisci_categorie(ptdemog, ['ETHNICITY', 'GENDER', 'MARRY', 'RACE'])

adnimerge["GENDER", "MARRY", "ETHNICITY", "RACE"]"""

def ricostruisci_categorie(df, lista_categorie):
    df = df.copy()
    tutte_le_dummy = []

    for prefix in lista_categorie:#["GENDER", "MARRY", "ETHNICITY", "RACE"]:
        cols = df.columns[df.columns.str.startswith(prefix)].tolist()
        tutte_le_dummy.extend(cols)   # accumula solo per il drop finale

        ricostruita = pd.from_dummies(df[cols], sep="_", default_category="__MISSING__").replace("__MISSING__", np.nan)
        df[prefix] = ricostruita[prefix]

    df = df.drop(columns=tutte_le_dummy)
    return df

In [7]:
lista_categorie = ["GENDER", "MARRY", "ETHNICITY", "RACE"]

adnimerge = ricostruisci_categorie(adnimerge, lista_categorie)
ptdemog = ricostruisci_categorie(ptdemog, lista_categorie)

In [8]:
adnimerge[['GENDER', 'MARRY', 'ETHNICITY', 'RACE']]


,GENDER,MARRY,ETHNICITY,RACE
0,1,1,0,5
1,1,1,0,5
2,1,1,0,5
3,1,1,0,5
4,1,1,0,5
...,...,...,...,...
9301,0,2,0,4
9302,1,1,0,5
9303,1,2,0,4
9304,1,2,0,4


In [9]:
adnimerge['AGE'] = adnimerge['AGE'].round(1)
ptdemog['AGE'] = ptdemog['AGE'].round(1)

## Merge

In [10]:
keys = ["RID", "EXAMDATE"]

# Conto delle combinazioni uniche di chiavi
left_keys = adnimerge[keys].drop_duplicates()
right_keys = ptdemog[keys].drop_duplicates()

key_match = left_keys.merge(
    right_keys,
    on=keys,
    how="outer",
    indicator=True
)

counts = key_match["_merge"].value_counts()
print("Conteggio chiavi uniche per [RID, EXAMDATE]:")
print(counts)

print(f"Match: {counts.get('both', 0)}")
print(f"Solo in adnimerge: {counts.get('left_only', 0)}")
print(f"Solo in ptdemog: {counts.get('right_only', 0)}")

Conteggio chiavi uniche per [RID, EXAMDATE]:
_merge
left_only     8861
right_only    5515
both           445
Name: count, dtype: int64
Match: 445
Solo in adnimerge: 8861
Solo in ptdemog: 5515


In [11]:
# --- 2. Merge su RID + EXAMDATE ---
merged = pd.merge(
    adnimerge,
    ptdemog,
    on=keys,
    how="outer",
    indicator=True
)

In [12]:
# --- 3. Log di controllo post-merge ---
print(merged["_merge"].value_counts())
#merged = merged.drop(columns="_merge") #tenere per le visualizazioni eliminare solo alla fine

_merge
left_only     8861
right_only    5515
both           445
Name: count, dtype: int64


In [13]:
merged = merged[sorted(merged.columns)]
merged

,ADAS11,ADAS13,AGE_bl,AGE_x,AGE_y,APOE4,CDRSB,COLPROT,DX,DX_0,...,RAVLT_immediate,RID,VISCODE_x,VISCODE_y,VISIT_MONTH_x,VISIT_MONTH_y,Ventricles,_merge,update_stamp_x,update_stamp_y
0,NaN,NaN,NaN,NaN,60.7,NaN,NaN,NaN,NaN,NaN,...,NaN,1,NaN,f,NaN,0.0,NaN,right_only,NaT,2005-08-18 00:00:00
1,NaN,NaN,NaN,NaN,74.4,NaN,NaN,NaN,NaN,NaN,...,NaN,2,NaN,sc,NaN,0.0,NaN,right_only,NaT,2005-08-17 00:00:00
2,10.67,18.67,74.3,74.3,NaN,0.0,0.0,ADNI1,NaN,1.0,...,44.0,2,bl,NaN,0.0,NaN,118233.0,left_only,2023-07-07 04:59:40,NaT
3,NaN,NaN,NaN,NaN,79.5,NaN,NaN,NaN,NaN,NaN,...,NaN,2,NaN,sc,NaN,61.0,NaN,right_only,NaT,2013-03-22 15:23:58
4,NaN,NaN,NaN,NaN,80.5,NaN,NaN,NaN,NaN,NaN,...,NaN,2,NaN,m72,NaN,73.0,NaN,right_only,NaT,2013-05-30 10:05:05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14816,NaN,NaN,NaN,NaN,74.3,NaN,NaN,NaN,NaN,NaN,...,NaN,10891,NaN,sc,NaN,0.0,NaN,right_only,NaT,2026-02-26 00:04:48
14817,NaN,NaN,NaN,NaN,76.6,NaN,NaN,NaN,NaN,NaN,...,NaN,10893,NaN,sc,NaN,0.0,NaN,right_only,NaT,2026-02-26 00:04:48
14818,NaN,NaN,NaN,NaN,79.6,NaN,NaN,NaN,NaN,NaN,...,NaN,10894,NaN,sc,NaN,0.0,NaN,right_only,NaT,2026-02-26 00:04:48
14819,NaN,NaN,NaN,NaN,67.4,NaN,NaN,NaN,NaN,NaN,...,NaN,10895,NaN,sc,NaN,0.0,NaN,right_only,NaT,2026-02-26 00:04:48


## Creazione colonna unificata

In [14]:
both_rows = merged[sorted(merged.columns)]
both_rows[both_rows["_merge"] == "both"]

,ADAS11,ADAS13,AGE_bl,AGE_x,AGE_y,APOE4,CDRSB,COLPROT,DX,DX_0,...,RAVLT_immediate,RID,VISCODE_x,VISCODE_y,VISIT_MONTH_x,VISIT_MONTH_y,Ventricles,_merge,update_stamp_x,update_stamp_y
77,2.0,4.0,72.6,77.6,77.7,0.0,0.0,ADNIGO,NaN,1.0,...,58.0,21,m60,sc,59.0,60.0,18783.0,both,2023-07-07 04:59:41,2014-07-10 19:03:08
78,3.0,5.0,72.6,78.6,78.7,0.0,0.0,ADNI2,NaN,1.0,...,53.0,21,m72,m72,72.0,72.0,22013.0,both,2023-07-07 04:59:41,2013-05-30 10:05:05
96,5.0,10.0,71.7,76.8,77.0,0.0,0.0,ADNIGO,NaN,1.0,...,42.0,23,m60,sc,61.0,62.0,28003.0,both,2023-07-07 04:59:41,2013-03-22 15:23:58
97,6.0,10.0,71.7,77.8,78.0,0.0,0.0,ADNI2,NaN,1.0,...,39.0,23,m72,m72,73.0,74.0,29028.0,both,2023-07-07 04:59:41,2013-05-30 10:05:05
124,5.0,10.0,77.7,82.8,82.9,0.0,0.0,ADNIGO,NaN,1.0,...,53.0,31,m60,sc,61.0,62.0,31653.0,both,2023-07-07 04:59:41,2014-01-15 19:03:02
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6833,8.0,12.0,70.2,71.2,71.3,0.0,1.0,ADNI2,NaN,0.0,...,32.0,2396,m12,m12,12.0,13.0,49461.0,both,2023-07-07 04:59:53,2013-05-30 10:05:05
6842,14.0,26.0,71.5,72.5,72.6,2.0,4.0,ADNI2,NaN,0.0,...,28.0,2398,m12,m12,12.0,13.0,59320.0,both,2023-07-07 04:59:53,2013-05-30 10:05:05
6851,14.0,24.0,79.1,80.1,80.3,0.0,4.5,ADNI2,NaN,0.0,...,26.0,2403,m12,m12,12.0,14.0,39370.0,both,2023-07-07 04:59:53,2013-05-30 10:05:05
6859,10.0,16.0,71.6,72.6,72.7,0.0,0.5,ADNI2,NaN,0.0,...,47.0,2405,m12,m12,12.0,13.0,24946.0,both,2023-07-07 04:59:53,2013-05-30 10:05:05


In [15]:
cols_x = sorted([c for c in both_rows.columns if c.endswith('_x')])
coppie = []
for c in cols_x:
    base = c[:-2]
    c_y = base + '_y'
    if c_y in both_rows.columns:
        coppie.append(base)

print("Coppie trovate:", coppie)

Coppie trovate: ['AGE', 'EDUCATION', 'ETHNICITY', 'GENDER', 'MARRY', 'RACE', 'VISCODE', 'VISIT_MONTH', 'update_stamp']


In [16]:
# --- 2. Confronta ogni coppia, con tolleranza SOLO per le colonne numeriche ---
for base in coppie:
    if base == "update_stamp":
        continue  # non considerare update_stamp per ora

    col_x, col_y = f"{base}_x", f"{base}_y"

    valid_mask = both_rows[col_x].notna() & both_rows[col_y].notna()
    if pd.api.types.is_numeric_dtype(both_rows[col_x]) and pd.api.types.is_numeric_dtype(both_rows[col_y]):
        diff_mask = valid_mask & ((both_rows[col_x] - both_rows[col_y]).abs() > 0.3)   # tolleranza per float
    else:
        diff_mask = valid_mask & (both_rows[col_x] != both_rows[col_y])                # confronto esatto per stringhe/categorie

    n_diff = diff_mask.sum()
    print(f"{base}: {n_diff} righe diverse su {len(both_rows)}")

    if n_diff > 0:
        print(both_rows.loc[diff_mask, ['RID', col_x, col_y]].head(10))
        print()

    # --- Coalesce: priorità a PTDEMOG (_y) solo per AGE, altrimenti priorità ad ADNIMERGE (_x) ---
    if base == "AGE":
        merged[base] = merged[col_y].combine_first(merged[col_x])
    else:
        merged[base] = merged[col_x].combine_first(merged[col_y])

AGE: 14 righe diverse su 14821
      RID  AGE_x  AGE_y
286    61   82.1   82.4
287    61   83.1   83.4
1160  260   83.6   83.9
1390  311   83.1   83.4
1842  420   77.4   77.8
1843  420   78.5   78.9
2126  498   74.5   74.9
2127  498   75.5   76.0
2542  602   76.1   76.4
3797  914   78.4   78.8

EDUCATION: 4 righe diverse su 14821
       RID  EDUCATION_x  EDUCATION_y
1631   376         15.0         14.0
2645   625          6.0          8.0
6011  2079         13.0         12.0
6371  2210         12.0         13.0

ETHNICITY: 1 righe diverse su 14821
       RID ETHNICITY_x ETHNICITY_y
6175  2146           1           0

GENDER: 0 righe diverse su 14821
MARRY: 17 righe diverse su 14821
      RID MARRY_x MARRY_y
96     23       3       1
97     23       3       1
982   214       1       3
1359  303       1       3
1390  311       1       3
1663  382       1       3
1796  413       1       3
2855  677       1       3
2884  680       1       3
4118  989       1       3

RACE: 2 righe diverse 

In [17]:
display_cols = ['RID', 'VISCODE', 'VISCODE_x', 'VISCODE_y', 'VISIT_MONTH', 'EXAMDATE', 'ETHNICITY','RACE', 'AGE', '_merge']
merged[merged["RID"] == 2][display_cols].head(20)

,RID,VISCODE,VISCODE_x,VISCODE_y,VISIT_MONTH,EXAMDATE,ETHNICITY,RACE,AGE,_merge
1,2,sc,NaN,sc,0.0,2005-08-17,0,5,74.4,right_only
2,2,bl,bl,NaN,0.0,2005-09-08,0,5,74.3,left_only
3,2,sc,NaN,sc,61.0,2010-09-22,0,5,79.5,right_only
4,2,m72,NaN,m72,73.0,2011-09-19,0,5,80.5,right_only


In [18]:
merged

,ADAS11,ADAS13,AGE_bl,AGE_x,AGE_y,APOE4,CDRSB,COLPROT,DX,DX_0,...,update_stamp_x,update_stamp_y,AGE,EDUCATION,ETHNICITY,GENDER,MARRY,RACE,VISCODE,VISIT_MONTH
0,NaN,NaN,NaN,NaN,60.7,NaN,NaN,NaN,NaN,NaN,...,NaT,2005-08-18 00:00:00,60.7,18.0,NaN,0,1,NaN,f,0.0
1,NaN,NaN,NaN,NaN,74.4,NaN,NaN,NaN,NaN,NaN,...,NaT,2005-08-17 00:00:00,74.4,16.0,0,1,1,5,sc,0.0
2,10.67,18.67,74.3,74.3,NaN,0.0,0.0,ADNI1,NaN,1.0,...,2023-07-07 04:59:40,NaT,74.3,16.0,0,1,1,5,bl,0.0
3,NaN,NaN,NaN,NaN,79.5,NaN,NaN,NaN,NaN,NaN,...,NaT,2013-03-22 15:23:58,79.5,16.0,0,1,3,5,sc,61.0
4,NaN,NaN,NaN,NaN,80.5,NaN,NaN,NaN,NaN,NaN,...,NaT,2013-05-30 10:05:05,80.5,16.0,0,1,3,5,m72,73.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14816,NaN,NaN,NaN,NaN,74.3,NaN,NaN,NaN,NaN,NaN,...,NaT,2026-02-26 00:04:48,74.3,20.0,0,1,1,5,sc,0.0
14817,NaN,NaN,NaN,NaN,76.6,NaN,NaN,NaN,NaN,NaN,...,NaT,2026-02-26 00:04:48,76.6,12.0,0,0,3,5,sc,0.0
14818,NaN,NaN,NaN,NaN,79.6,NaN,NaN,NaN,NaN,NaN,...,NaT,2026-02-26 00:04:48,79.6,14.0,0,1,3,5,sc,0.0
14819,NaN,NaN,NaN,NaN,67.4,NaN,NaN,NaN,NaN,NaN,...,NaT,2026-02-26 00:04:48,67.4,16.0,0,0,1,5,sc,0.0


## Controllo missing data

In [19]:
missing_report = merged.isna().sum().sort_values(ascending=False)
print(missing_report)

PTADBEG            14445
DX                 14302
HAS_QC_ERROR       13543
PTCOGBEG           12295
RACE_y              8978
ETHNICITY_y         8920
MARRY_y             8905
EDUCATION_y         8882
VISCODE_y           8863
AGE_y               8862
VISIT_MONTH_y       8862
PTDOB               8861
GENDER_y            8861
update_stamp_y      8861
PTID                8861
Fusiform            7075
MidTemp             7075
Entorhinal          7075
Hippocampus         6684
Ventricles          6094
APOE4               5707
ADAS13              5641
FAQ                 5621
RAVLT_immediate     5611
CDRSB               5601
ADAS11              5565
ETHNICITY_x         5561
MARRY_x             5553
MMSE                5544
RACE_x              5530
AGE_x               5521
AGE_bl              5521
GENDER_x            5515
FSVERSION           5515
COLPROT             5515
DX_1                5515
DX_0                5515
EDUCATION_x         5515
DX_2                5515
update_stamp_x      5515


In [20]:
missing_count = merged.isna().sum()
missing_pct = (missing_count / len(merged) * 100).round(2)

missing_report = pd.DataFrame({
    "n_missing": missing_count,
    "pct_missing": missing_pct
}).sort_values("n_missing", ascending=False)

print(missing_report)

                 n_missing  pct_missing
PTADBEG              14445        97.46
DX                   14302        96.50
HAS_QC_ERROR         13543        91.38
PTCOGBEG             12295        82.96
RACE_y                8978        60.58
ETHNICITY_y           8920        60.18
MARRY_y               8905        60.08
EDUCATION_y           8882        59.93
VISCODE_y             8863        59.80
AGE_y                 8862        59.79
VISIT_MONTH_y         8862        59.79
PTDOB                 8861        59.79
GENDER_y              8861        59.79
update_stamp_y        8861        59.79
PTID                  8861        59.79
Fusiform              7075        47.74
MidTemp               7075        47.74
Entorhinal            7075        47.74
Hippocampus           6684        45.10
Ventricles            6094        41.12
APOE4                 5707        38.51
ADAS13                5641        38.06
FAQ                   5621        37.93
RAVLT_immediate       5611        37.86


In [21]:
merged.dtypes

ADAS11                    float64
ADAS13                    float64
AGE_bl                    float64
AGE_x                     float64
AGE_y                     float64
APOE4                     float64
CDRSB                     float64
COLPROT                       str
DX                        float64
DX_0                      float64
DX_1                      float64
DX_2                      float64
EDUCATION_x               float64
EDUCATION_y               float64
ETHNICITY_x                   str
ETHNICITY_y                   str
EXAMDATE           datetime64[us]
Entorhinal                float64
FAQ                       float64
FSVERSION                 float64
Fusiform                  float64
GENDER_x                      str
GENDER_y                      str
HAS_QC_ERROR              float64
Hippocampus               float64
ICV                       float64
IMAGEUID                  float64
MARRY_x                       str
MARRY_y                       str
MMSE          

## Age arrotondato a un decimale

In [22]:
merged["AGE"] = merged["AGE"].round(1)

In [23]:
display_cols = ['RID', 'VISCODE', 'VISIT_MONTH', 'EXAMDATE', 'ETHNICITY','RACE', 'AGE', '_merge']
merged[display_cols].head(20)

,RID,VISCODE,VISIT_MONTH,EXAMDATE,ETHNICITY,RACE,AGE,_merge
0,1,f,0.0,2005-08-18,NaN,NaN,60.7,right_only
1,2,sc,0.0,2005-08-17,0,5,74.4,right_only
2,2,bl,0.0,2005-09-08,0,5,74.3,left_only
3,2,sc,61.0,2010-09-22,0,5,79.5,right_only
4,2,m72,73.0,2011-09-19,0,5,80.5,right_only
5,3,sc,0.0,2005-08-18,0,5,81.3,right_only
6,3,bl,0.0,2005-09-12,0,5,81.3,left_only
7,3,m06,6.0,2006-03-13,0,5,81.8,left_only
8,3,m12,12.0,2006-09-12,0,5,82.3,left_only
9,3,m24,24.0,2007-09-12,0,5,83.3,left_only


## Ricalcolo "VISCODE" 

In [24]:
def recompute_visit_month(df, id_col="RID", date_col="EXAMDATE", viscode_col="VISCODE", out_col="VISIT_MONTH"):
    """Ricalcola VISIT_MONTH come mesi trascorsi dalla baseline (bl), per ogni RID."""
    df = df.copy()

    # data di riferimento (bl) per ogni soggetto
    baseline_dates = (
        df.loc[df[viscode_col] == "bl", [id_col, date_col]]
        .drop_duplicates(subset=id_col)
        .set_index(id_col)[date_col]
    )

    df["_baseline_ref"] = df[id_col].map(baseline_dates)
    delta_days = (df[date_col] - df["_baseline_ref"]).dt.days
    df[out_col] = (delta_days / 30.4375).round(0)   # media giorni/mese

    df = df.drop(columns="_baseline_ref")
    return df

In [25]:
def add_visit_month(df, id_col, date_col):
    """VISIT_MONTH = mesi trascorsi dalla prima visita (baseline) del paziente."""
    df = df.copy().sort_values([id_col, date_col])
    baseline = df.groupby(id_col)[date_col].transform("min")
    df["VISIT_MONTH"] = ((df[date_col] - baseline).dt.days / 30.44).round().astype("Int64")
    return df

In [26]:
merged = add_visit_month(merged, "RID", "EXAMDATE")

## Sistemare "RID" dove manca la "bl"
problema, soggetti senza "bl" su "VISCODE"

In [30]:
display_cols = ['RID', 'VISCODE', 'VISIT_MONTH', 'EXAMDATE', 'ETHNICITY','RACE', 'AGE', '_merge']
merged[merged["RID"] == 4003][display_cols].head(20)

,RID,VISCODE,VISIT_MONTH,EXAMDATE,ETHNICITY,RACE,AGE,_merge
6873,4003,sc,0,2011-02-22,NaN,5,72.3,right_only
6874,4003,m06,7,2011-10-07,NaN,5,72.8,left_only
6875,4003,m12,14,2012-04-19,NaN,5,73.3,left_only
6876,4003,m24,26,2013-05-08,NaN,5,74.4,left_only
6877,4003,m36,38,2014-04-15,NaN,5,75.3,left_only
6878,4003,m60,62,2016-04-19,NaN,5,77.3,left_only
6879,4003,m78,81,2017-11-29,NaN,5,78.9,left_only
6880,4003,m102,104,2019-11-05,NaN,5,80.9,left_only


In [29]:
display_cols = ['RID', 'VISCODE', 'VISIT_MONTH', 'EXAMDATE', 'ETHNICITY','RACE', 'AGE', '_merge']
merged[merged["RID"] == 1226][display_cols].head(20)

,RID,VISCODE,VISIT_MONTH,EXAMDATE,ETHNICITY,RACE,AGE,_merge
5049,1226,sc,0,2007-01-08,0,5,82.6,right_only
5050,1226,m06,8,2007-08-27,0,5,83.1,left_only
5051,1226,m24,25,2009-02-17,0,5,84.6,left_only
5052,1226,m36,39,2010-03-30,0,5,85.7,left_only
5053,1226,m48,49,2011-02-09,0,5,86.7,both


In [38]:
merged[merged["AGE_bl"].isna()][["RID", "VISCODE", "AGE_bl"]]

,RID,VISCODE,AGE_bl
0,1,f,NaN
1,2,sc,NaN
3,2,sc,NaN
4,2,m72,NaN
5,3,sc,NaN
...,...,...,...
14816,10891,sc,NaN
14817,10893,sc,NaN
14818,10894,sc,NaN
14819,10895,sc,NaN


In [39]:
soggetti_senza_bl = (
    merged.groupby("RID")["VISCODE"]
    .apply(lambda x: "bl" not in x.values)
)

soggetti_senza_bl = soggetti_senza_bl[soggetti_senza_bl].index

soggetti_senza_bl

Index([    1,     9,    11,    13,    17,    18,    20,    24,    25,    26,
       ...
       10885, 10887, 10888, 10889, 10890, 10891, 10893, 10894, 10895, 10898],
      dtype='int64', name='RID', length=2508)

In [40]:
mask = merged["RID"].isin(soggetti_senza_bl) & (merged["VISCODE"] == "sc")

n_da_correggere = mask.sum()
print(f"Righe da correggere (sc -> bl): {n_da_correggere}")

merged.loc[mask, "VISCODE"] = "bl"

Righe da correggere (sc -> bl): 1944


In [41]:
check_bl = merged[merged["RID"].isin(soggetti_senza_bl) & (merged["VISCODE"] == "bl")].groupby("RID").size()
print(check_bl)

rid_senza_bl = set(soggetti_senza_bl) - set(check_bl.index)
print("RID nella lista senza VISCODE=bl dopo la correzione:", rid_senza_bl)

RID
460      1
542      1
662      1
2001     1
2004     1
        ..
10891    1
10893    1
10894    1
10895    1
10898    1
Length: 1944, dtype: int64
RID nella lista senza VISCODE=bl dopo la correzione: {1, 9, 11, 13, 17, 18, 20, 24, 25, 26, 27, 28, 32, 34, 36, 37, 39, 49, 52, 62, 63, 64, 65, 71, 73, 79, 82, 85, 92, 99, 100, 104, 114, 115, 117, 119, 122, 124, 131, 132, 133, 134, 136, 137, 143, 144, 145, 146, 148, 151, 152, 153, 154, 157, 163, 164, 165, 170, 174, 175, 180, 181, 185, 189, 192, 193, 197, 198, 199, 201, 202, 203, 206, 207, 209, 211, 212, 215, 218, 220, 224, 226, 233, 234, 235, 236, 237, 238, 239, 242, 244, 246, 247, 248, 250, 251, 252, 253, 254, 255, 261, 263, 264, 265, 267, 268, 270, 271, 274, 275, 277, 278, 279, 280, 281, 287, 297, 302, 305, 306, 308, 309, 317, 318, 320, 322, 323, 329, 330, 333, 334, 338, 340, 342, 345, 346, 347, 348, 349, 350, 353, 355, 357, 358, 364, 365, 367, 368, 371, 373, 375, 379, 380, 381, 383, 385, 387, 395, 396, 398, 399, 402, 411, 412, 415, 4

In [42]:
display_cols = ['RID', 'VISCODE', 'VISIT_MONTH', 'EXAMDATE', 'ETHNICITY','RACE', 'AGE', '_merge']
merged[merged["RID"] == 1226][display_cols].head(20)

,RID,VISCODE,VISIT_MONTH,EXAMDATE,ETHNICITY,RACE,AGE,_merge
5049,1226,bl,0,2007-01-08,0,5,82.6,right_only
5050,1226,m06,8,2007-08-27,0,5,83.1,left_only
5051,1226,m24,25,2009-02-17,0,5,84.6,left_only
5052,1226,m36,39,2010-03-30,0,5,85.7,left_only
5053,1226,m48,49,2011-02-09,0,5,86.7,both


Ricerca dati mancanti su "ETHNCITY"

In [56]:
display_cols = ['RID', 'ETHNICITY','RACE', 'VISCODE', '_merge']
merged[merged["ETHNICITY"] .isna()][display_cols].head(20)

,RID,ETHNICITY,RACE,VISCODE,_merge
0,1,NaN,NaN,f,right_only
834,175,NaN,NaN,f,right_only
922,198,NaN,5,f,right_only
935,201,NaN,NaN,f,right_only
1039,228,NaN,5,sc,right_only
1040,228,NaN,5,bl,left_only
1111,253,NaN,NaN,f,right_only
1113,255,NaN,NaN,f,right_only
1367,306,NaN,NaN,f,right_only
1379,309,NaN,NaN,f,right_only


In [45]:
merged[merged["ETHNICITY"].isna()][["RID", "VISCODE", "ETHNICITY"]]


,RID,VISCODE,ETHNICITY
0,1,f,NaN
834,175,f,NaN
922,198,f,NaN
935,201,f,NaN
1039,228,sc,NaN
...,...,...,...
14037,7119,bl,NaN
14114,10068,bl,NaN
14388,10354,bl,NaN
14757,10827,bl,NaN


In [53]:
merged[merged["VISCODE"].isna()][["RID", "VISCODE", "VISIT_MONTH", "RACE", "_merge"]]

,RID,VISCODE,VISIT_MONTH,RACE,_merge
13776,6970,NaN,40,5,right_only
14027,7112,NaN,14,4,right_only


In [52]:
display_cols = ['RID', 'VISCODE','VISIT_MONTH', 'AGE', '_merge']
merged[merged["RID"] == 7112][display_cols].head(20)

,RID,VISCODE,VISIT_MONTH,AGE,_merge
14026,7112,bl,0,57.4,right_only
14027,7112,NaN,14,58.5,right_only


In [55]:
print(ptdemog.loc[ptdemog["RID"].isin([6970, 7112]), ["RID", "EXAMDATE", "VISCODE"]])

       RID   EXAMDATE VISCODE
4971  6970 2021-06-16      sc
4972  6970 2024-10-08     NaN
5166  7112 2022-10-12      sc
5167  7112 2023-11-29     NaN


## update_stamp unicone 
per data più aggiornata

In [57]:
def most_recent_timestamp(df, col_x, col_y, out_col):
    """Estrae, riga per riga, la data più recente tra col_x e col_y.

    Se una delle due è NaN, usa l'altra automaticamente (max ignora i NaN
    salvo che siano entrambe NaN, nel qual caso il risultato resta NaN).
    """
    df = df.copy()
    df[out_col] = df[[col_x, col_y]].max(axis=1)
    return df

In [58]:
merged = most_recent_timestamp(merged, "update_stamp_x", "update_stamp_y", "update_stamp")

In [59]:
print(merged[["update_stamp_x", "update_stamp_y", "update_stamp"]].head(10))
print("NaN nella colonna finale:", merged["update_stamp"].isna().sum())

       update_stamp_x      update_stamp_y        update_stamp
0                 NaT 2005-08-18 00:00:00 2005-08-18 00:00:00
1                 NaT 2005-08-17 00:00:00 2005-08-17 00:00:00
2 2023-07-07 04:59:40                 NaT 2023-07-07 04:59:40
3                 NaT 2013-03-22 15:23:58 2013-03-22 15:23:58
4                 NaT 2013-05-30 10:05:05 2013-05-30 10:05:05
5                 NaT 2005-08-18 00:00:00 2005-08-18 00:00:00
6 2023-07-07 04:59:40                 NaT 2023-07-07 04:59:40
7 2023-07-07 04:59:40                 NaT 2023-07-07 04:59:40
8 2023-07-07 04:59:40                 NaT 2023-07-07 04:59:40
9 2023-07-07 04:59:40                 NaT 2023-07-07 04:59:40
NaN nella colonna finale: 0


In [61]:
display_cols = ['RID', 'VISCODE','update_stamp_x', 'update_stamp_y', 'update_stamp', '_merge']
merged[merged["_merge"] == "both"][display_cols]

,RID,VISCODE,update_stamp_x,update_stamp_y,update_stamp,_merge
77,21,m60,2023-07-07 04:59:41,2014-07-10 19:03:08,2023-07-07 04:59:41,both
78,21,m72,2023-07-07 04:59:41,2013-05-30 10:05:05,2023-07-07 04:59:41,both
96,23,m60,2023-07-07 04:59:41,2013-03-22 15:23:58,2023-07-07 04:59:41,both
97,23,m72,2023-07-07 04:59:41,2013-05-30 10:05:05,2023-07-07 04:59:41,both
124,31,m60,2023-07-07 04:59:41,2014-01-15 19:03:02,2023-07-07 04:59:41,both
...,...,...,...,...,...,...
6833,2396,m12,2023-07-07 04:59:53,2013-05-30 10:05:05,2023-07-07 04:59:53,both
6842,2398,m12,2023-07-07 04:59:53,2013-05-30 10:05:05,2023-07-07 04:59:53,both
6851,2403,m12,2023-07-07 04:59:53,2013-05-30 10:05:05,2023-07-07 04:59:53,both
6859,2405,m12,2023-07-07 04:59:53,2013-05-30 10:05:05,2023-07-07 04:59:53,both
